In [ ]:
# hide
import numpy as np
import pyquist as pq
from pyquist.helper import pitch_name_to_pitch


def resample(x: pq.Audio, ratio: float) -> pq.Audio:
    """Read x at `ratio` times its original speed, with linear interpolation."""
    samples = np.asarray(x.samples)
    N = len(samples)
    M = int(N / ratio)                        # the duration scales by 1 / ratio
    p = np.arange(M) * ratio                  # fractional read positions in x
    p_floor = np.floor(p).astype(int)
    alpha = (p - p_floor)[:, np.newaxis]
    p_next = np.minimum(p_floor + 1, N - 1)
    y = (1 - alpha) * samples[p_floor] + alpha * samples[p_next]
    return pq.Audio(y, x.sample_rate)


def repitch(x: pq.Audio, k: float) -> pq.Audio:
    """Shift the pitch of x by k steps."""
    return resample(x, 2 ** (k / 12))

In [ ]:
LOOP_START, LOOP_END = 0.586803, 0.766916   # seconds: audio samples 25,878 and 33,821
RELEASE = 3.321361                          # seconds: audio sample 146,472, where we jump to the release


def clarinet(pitch: str, duration: float) -> pq.Audio:
    """A sampled clarinet with independent control over pitch and duration."""
    recording = pq.Audio.from_file("../assets/clarinet.wav")
    x, f_s = np.asarray(recording.samples), recording.sample_rate
    k = pitch_name_to_pitch(pitch) - pitch_name_to_pitch("A#5")
    # Loop first, at the original pitch, so the loop points are exact audio samples. Repitching
    # will then scale the duration by 2^(-k/12), so we loop to duration * 2^(k/12) seconds.
    target = duration * 2 ** (k / 12) * f_s
    s, e, r = (round(t * f_s) for t in (LOOP_START, LOOP_END, RELEASE))
    # Play up to the loop, repeat the loop to fill out the duration, then jump to the release
    n = max(0, round((target - s - (len(x) - r)) / (e - s)))
    looped = pq.Audio(np.concatenate([x[:s]] + [x[s:e]] * n + [x[r:]]), f_s)
    return repitch(looped, k)


# The same pitch at three durations, then three pitches at the same duration
notes = [("A#5", 2.5), ("A#5", 4.0), ("A#5", 8.0), ("D5", 4.0), ("F5", 4.0), ("D6", 4.0)]
onsets = np.cumsum([0.0] + [duration for _, duration in notes[:-1]])
melody = pq.Score([(t, {"pitch": pitch, "duration": duration})
                   for t, (pitch, duration) in zip(onsets, notes)])
pq.play(melody.render(clarinet), normalize=True)